# DT evaluation – cycle 100 retrospectiveThis notebook loads Decision Transformer evaluation metrics from `.goalie/metrics_log.jsonl` and reward preset summaries from `compare_presets.py` to support a governance-focused retrospective for production cycle 100 (or the nearest available cycle).

In [ ]:
from __future__ import annotationsimport json, subprocess, sysfrom pathlib import Pathimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as sns# Project / path setupROOT = Path.cwd()if (ROOT / 'investing' / 'agentic-flow').is_dir():    PROJECT_ROOT = ROOT / 'investing' / 'agentic-flow'    ROOT_GOALIE = ROOT / '.goalie'else:    PROJECT_ROOT = ROOT    ROOT_GOALIE = PROJECT_ROOT / '.goalie'sys.path.append(str(PROJECT_ROOT))from scripts.analysis import dt_evaluation_dashboard as dt_dashGOALIE_DIR = PROJECT_ROOT / '.goalie'METRICS_LOG = GOALIE_DIR / 'metrics_log.jsonl'TRAJ_CYCLE_100 = ROOT_GOALIE / 'trajectories_cycle_100.jsonl'TRAJ_DEFAULT = ROOT_GOALIE / 'trajectories.jsonl'TRAJECTORIES = TRAJ_CYCLE_100 if TRAJ_CYCLE_100.is_file() and TRAJ_CYCLE_100.stat().st_size > 0 else TRAJ_DEFAULTTRAJECTORIES

In [ ]:
sns.set(style='whitegrid')events = dt_dash.read_dt_events(    METRICS_LOG,    since=None,    until=None,    checkpoint_pattern=None,    compare=None,)print(f'Loaded {len(events)} dt_evaluation events from {METRICS_LOG}')if not events:    raise RuntimeError('No dt_evaluation events found; run `af evaluate-dt` first.')# Flatten events into DataFramesrows, per_circle_rows = [], []for ev in events:    rows.append({        'timestamp': ev.timestamp,        'checkpoint': ev.checkpoint,        'checkpoint_name': ev.checkpoint_name,        'run_name': ev.run_name,        'top1': ev.top1,        'top3': ev.top3,        'cont_mae': ev.cont_mae,        'cont_mse': ev.cont_mse,        'total_positions': ev.total_positions,    })    for circle, acc in ev.per_circle_top1.items():        per_circle_rows.append({            'timestamp': ev.timestamp,            'checkpoint': ev.checkpoint,            'circle': circle,            'top1': acc,        })eval_df = pd.DataFrame(rows).sort_values('timestamp')per_circle_df = pd.DataFrame(per_circle_rows).sort_values('timestamp')eval_df.head()

In [ ]:
# Global DT evaluation trends: top1 accuracy and MAE over timefig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)axes[0].plot(eval_df['timestamp'], eval_df['top1'], marker='o')axes[0].set_ylabel('top1_accuracy')axes[0].set_title('DT evaluation: top1 accuracy over time')mae_df = eval_df.dropna(subset=['cont_mae'])axes[1].plot(mae_df['timestamp'], mae_df['cont_mae'], marker='o', color='orange')axes[1].set_ylabel('MAE')axes[1].set_title('Continuous head MAE over time')plt.xticks(rotation=45)plt.tight_layout()top1_stats = dt_dash.summarize_metric(eval_df['top1'])cont_mae_stats = dt_dash.summarize_metric(mae_df['cont_mae'])per_circle_medians = per_circle_df.groupby('circle')['top1'].median().to_dict()recs = dt_dash.compute_threshold_recommendations(top1_stats, cont_mae_stats, per_circle_medians)staging_rate = dt_dash.pass_rate(events, recs.get('staging', {}))prod_rate = dt_dash.pass_rate(events, recs.get('production', {}))recs, staging_rate, prod_rate

In [ ]:
# BML cycles: locate cycle 100 window and latency p95 per cyclebml_rows = []if METRICS_LOG.is_file():    with METRICS_LOG.open('r', encoding='utf-8') as f:        for line in f:            line = line.strip()            if not line:                continue            obj = json.loads(line)            if 'cycle_index' not in obj or obj.get('type') not in {'action', 'reward', 'state'}:                continue            ts = obj.get('timestamp') or ''            dt = dt_dash.parse_time(ts)            if dt is None:                continue            row = {                'timestamp': dt,                'type': obj.get('type'),                'cycle_index': int(obj.get('cycle_index')),            }            if obj.get('type') == 'reward':                row['duration_ms'] = obj.get('duration_ms')                row['status'] = obj.get('status')            bml_rows.append(row)bml_df = pd.DataFrame(bml_rows)max_cycle = int(bml_df['cycle_index'].max()) if not bml_df.empty else Nonecycle = 100 if max_cycle and max_cycle >= 100 else max_cycleprint(f'Max cycle in metrics_log: {max_cycle}; using cycle={cycle} for retrospective')cycle_df = bml_df[bml_df['cycle_index'] == cycle]cycle_start = cycle_df['timestamp'].min()cycle_end = cycle_df['timestamp'].max()print('Cycle window:', cycle_start, '->', cycle_end)latency = bml_df[bml_df['type'] == 'reward'].dropna(subset=['duration_ms'])p95_by_cycle = (    latency.groupby('cycle_index')['duration_ms'].quantile(0.95).reset_index(name='p95_duration_ms'))plt.figure(figsize=(8, 4))plt.plot(p95_by_cycle['cycle_index'], p95_by_cycle['p95_duration_ms'], marker='o')plt.axvline(cycle, color='red', linestyle='--', label=f'cycle {cycle}')plt.xlabel('cycle_index')plt.ylabel('p95 duration_ms')plt.title('Latency p95 per cycle')plt.legend()plt.tight_layout()

In [ ]:
# Reward preset comparison (JSON) and reward-threshold curves per presetcompare_script = PROJECT_ROOT / 'scripts' / 'analysis' / 'compare_presets.py'cmd = [sys.executable, str(compare_script), '--trajectories', str(TRAJECTORIES), '--format', 'json']print('Running:', ' '.join(str(c) for c in cmd))res = subprocess.run(cmd, check=True, capture_output=True, text=True)compare_data = json.loads(res.stdout)metadata = compare_data['metadata']presets = compare_data['presets']print('Total steps after filtering:', metadata.get('total_steps'))rows = []for p in presets:    row = {'preset_name': p['name']}    for k, v in (p.get('reward_stats') or {}).items():        row[f'reward_{k}'] = v    sc = p.get('status_counts') or {}    row['success_count'] = sc.get('success', 0)    row['failure_count'] = sc.get('failure', 0)    dur_all = (p.get('duration_stats') or {}).get('all') or {}    row['duration_mean_ms'] = dur_all.get('mean')    rows.append(row)preset_df = pd.DataFrame(rows)preset_df# Compute per-step reward.value for each preset to plot threshold vs success rate curvestraj_rows = []if TRAJECTORIES.is_file() and TRAJECTORIES.stat().st_size > 0:    import importlib.util    bt_path = PROJECT_ROOT / 'scripts' / 'analysis' / 'build_trajectories.py'    spec = importlib.util.spec_from_file_location('build_trajectories_mod', bt_path)    bt_mod = importlib.util.module_from_spec(spec)    spec.loader.exec_module(bt_mod)  # type: ignore[arg-type]    REWARD_PRESETS = bt_mod.REWARD_PRESETS    compute_reward_value = bt_mod.compute_reward_value    with TRAJECTORIES.open('r', encoding='utf-8') as f:        for line in f:            line = line.strip()            if not line:                continue            rec = json.loads(line)            reward = rec.get('reward') or {}            status = reward.get('status')            duration_ms = reward.get('duration_ms')            if status is None or duration_ms is None:                continue            for name, (max_d, alpha) in REWARD_PRESETS.items():                val = compute_reward_value(status=status, duration_ms=float(duration_ms), max_duration_ms=max_d, alpha=alpha)                traj_rows.append({                    'cycle_index': rec.get('cycle_index'),                    'preset': name,                    'reward_value': val,                })traj_df = pd.DataFrame(traj_rows)if not traj_df.empty:    thresholds = [round(t, 2) for t in [x / 20.0 for x in range(1, 20)]]  # 0.05..0.95    curve_rows = []    for preset in sorted(traj_df['preset'].unique()):        sub = traj_df[traj_df['preset'] == preset]        for thr in thresholds:            frac = float((sub['reward_value'] >= thr).mean())            curve_rows.append({'preset': preset, 'threshold': thr, 'success_rate': frac})    curve_df = pd.DataFrame(curve_rows)    plt.figure(figsize=(8, 4))    for preset, sub in curve_df.groupby('preset'):        plt.plot(sub['threshold'], sub['success_rate'], marker='o', label=preset)    plt.xlabel('reward threshold')    plt.ylabel('fraction of steps with reward >= threshold')    plt.title('Reward threshold vs success rate by preset')    plt.legend()    plt.tight_layout()else:    print('No trajectories found for reward threshold analysis; ensure trajectories.jsonl is populated.')